#### Imports

In [14]:
import pandas as pd
import numpy as np
import random

np.random.seed(42)
random.seed(42)

df_events = pd.read_csv("../data/shipment_events.csv")
print(f"Events loaded: {len(df_events)}")

Events loaded: 8000


#### False positive suppression rules

In [15]:
# These are the REAL conditions under which a chargeback
# should be suppressed — supplier is not at fault
FP_RULES = {
    "ASN_LATE":      "EDI_System_Delay",
    "ASN_MISSING":   "Supplier_Portal_Outage",
    "PKG_DEVIATION": "Pre_Approved_Alternate",
    "QTY_MISMATCH":  "Transit_Damage",
    "OTIF_MISS":     "Weather_Force_Majeure",
    "LABEL_ERROR":   "Label_Reprinted_Before_Scan",
    "PO_VIOLATION":  "Tesla_Emergency_Reroute",
    "SHORTSHIP":     "Authorized_Split_Shipment",
}

# 20% of violations are actually false positives
# High-risk suppliers get slightly fewer FP suppression (they get less benefit of doubt)
FP_PROB = {
    "Low":    0.25,
    "Medium": 0.20,
    "High":   0.12,
}

print("FP rules ready.")

FP rules ready.


#### Apply false positive detection

In [16]:
def detect_false_positive(row):
    # Only applicable if there's a violation
    if not row["has_violation"]:
        return False, None
    
    fp_prob = FP_PROB[row["risk_profile"]]
    if random.random() < fp_prob:
        return True, FP_RULES[row["violation_type"]]
    
    return False, None

# Apply to every row
fp_results = df_events.apply(detect_false_positive, axis=1)

df_events["is_false_positive"]     = [r[0] for r in fp_results]
df_events["fp_suppression_reason"] = [r[1] for r in fp_results]

# A chargeback is only ISSUED if there's a violation AND it's NOT a false positive
df_events["chargeback_issued"] = (
    df_events["has_violation"] & ~df_events["is_false_positive"]
)

# If false positive — zero out the penalty
df_events.loc[df_events["is_false_positive"], "penalty_usd"] = 0.0

print("=== FALSE POSITIVE RESULTS ===")
total_violations  = df_events["has_violation"].sum()
total_fp          = df_events["is_false_positive"].sum()
total_chargebacks = df_events["chargeback_issued"].sum()
total_penalty     = df_events["penalty_usd"].sum()
fp_rate           = total_fp / total_violations

print(f"Total violations    : {total_violations:,}")
print(f"False positives     : {total_fp:,}  ({fp_rate:.1%} suppressed)")
print(f"Chargebacks issued  : {total_chargebacks:,}")
print(f"Total penalty value : ${total_penalty:,.0f}")

=== FALSE POSITIVE RESULTS ===
Total violations    : 1,159
False positives     : 197  (17.0% suppressed)
Chargebacks issued  : 962
Total penalty value : $4,992,441


#### Add dispute tracking

In [17]:
def assign_dispute(row):
    # ~28% of issued chargebacks get disputed by the supplier
    if not row["chargeback_issued"]:
        return False, "N/A", row["penalty_usd"]
    
    if random.random() < 0.28:
        status = random.choices(
            ["Upheld", "Reversed", "Partial_Credit", "Pending"],
            weights=[0.52, 0.22, 0.16, 0.10]
        )[0]
        
        # Adjust penalty based on dispute outcome
        if status == "Reversed":
            adjusted_penalty = 0.0
        elif status == "Partial_Credit":
            adjusted_penalty = round(row["penalty_usd"] * 0.50, 2)
        else:
            adjusted_penalty = row["penalty_usd"]
        
        return True, status, adjusted_penalty
    
    return False, "N/A", row["penalty_usd"]

dispute_results = df_events.apply(assign_dispute, axis=1)

df_events["dispute_raised"]   = [r[0] for r in dispute_results]
df_events["dispute_status"]   = [r[1] for r in dispute_results]
df_events["final_penalty_usd"]= [r[2] for r in dispute_results]

print("=== DISPUTE SUMMARY ===")
disputed = df_events[df_events["dispute_raised"]]
print(disputed["dispute_status"].value_counts())
print(f"\nTotal disputes raised : {len(disputed):,}")
print(f"Final penalty value   : ${df_events['final_penalty_usd'].sum():,.0f}")

=== DISPUTE SUMMARY ===
dispute_status
Upheld            125
Reversed           46
Pending            37
Partial_Credit     33
Name: count, dtype: int64

Total disputes raised : 241
Final penalty value   : $4,733,836


#### Save

In [18]:
df_events.to_csv("../data/shipment_events.csv", index=False)
print(f"Saved with chargeback logic: {df_events.shape}")
print(f"Columns: {df_events.columns.tolist()}")

Saved with chargeback logic: (8000, 22)
Columns: ['shipment_id', 'supplier_id', 'supplier_name', 'region', 'primary_plant', 'tier', 'part_category', 'risk_profile', 'ship_date', 'ship_month', 'ship_quarter', 'po_value_usd', 'qty_ordered', 'has_violation', 'violation_type', 'penalty_usd', 'is_false_positive', 'fp_suppression_reason', 'chargeback_issued', 'dispute_raised', 'dispute_status', 'final_penalty_usd']
